# Netflix Stock - Exploratory Data Analysis (EDA)

An analysis of Netflix (NFLX) stock prices from 2002 to 2026.

---

## Dataset Description

This dataset contains **daily Netflix (NFLX) stock market data** from **2002 to 2026**.

- **Source:** Yahoo Finance
- **Records:** 5,961 trading days
- **Main features:** `Open`, `High`, `Low`, `Close`, `Volume`
- **Time index:** Trading date (`Date`)

### 
- `Open`: Price at market open
- `High`: Highest price during the day
- `Low`: Lowest price during the day
- `Close`: Price at market close
- `Volume`: Number of shares traded

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded!')

---
## 2. Load and Inspect Data

In [ ]:
# Load dataset
df = pd.read_csv('datasets/netflix_stock.csv', header=[0, 1], index_col=0)
df.columns = df.columns.get_level_values(0)
df.index = pd.to_datetime(df.index)
df.index.name = 'Date'

print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')

### 2.1 First Rows

In [ ]:
df.head()

### 2.2 Last Rows

In [ ]:
df.tail()

### 2.3 Data Info

In [ ]:
df.info()

### 2.4 Statistical Summary

In [ ]:
df.describe()

---
## 3. Data Quality Check

In [ ]:
# Missing values
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
# Duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Date range
print(f'Start date: {df.index.min()}')
print(f'End date: {df.index.max()}')
print(f'Total days: {len(df)}')

---
## 4. Basic Visualizations

### 4.1 Stock Price Over Time

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], color='red')
plt.title('Netflix Stock Price (2002-2026)')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)
plt.show()

### 4.2 Trading Volume

In [ ]:
plt.figure(figsize=(14, 4))
plt.bar(df.index, df['Volume'] / 1e6, color='steelblue', width=2)
plt.title('Trading Volume Over Time')
plt.xlabel('Date')
plt.ylabel('Volume (Millions)')
plt.grid(True, alpha=0.3)
plt.show()

### 4.3 Moving Averages

Moving averages smooth out price data to show trends more clearly.

In [ ]:
# Calculate moving averages
df['MA_20'] = df['Close'].rolling(window=20).mean()
df['MA_50'] = df['Close'].rolling(window=50).mean()

df['MA_100'] = df['Close'].rolling(window=100).mean()

# Plot price with moving averages (last 5 years for clarity)
recent = df[df.index >= '2021-01-01']

plt.figure(figsize=(14, 5))
plt.plot(recent.index, recent['Close'], label='Close', color='black', alpha=0.7)
plt.plot(recent.index, recent['MA_20'], label='20-day MA', color='blue')
plt.plot(recent.index, recent['MA_50'], label='50-day MA', color='orange')
plt.plot(recent.index, recent['MA_100'], label='100-day MA', color='green')
plt.title('Netflix Price with Moving Averages (2021-2026)')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 5. Daily Returns Analysis

In [ ]:
# Calculate daily returns (percentage change)
df['Daily_Return'] = df['Close'].pct_change() * 100
df['Daily_Return'].head(10)

### 5.1 Returns Distribution (Histogram)

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['Daily_Return'].dropna(), bins=50, color='steelblue', edgecolor='white')
plt.axvline(df['Daily_Return'].mean(), color='red', linestyle='--', label=f'Mean: {df["Daily_Return"].mean():.2f}%')
plt.title('Distribution of Daily Returns')
plt.xlabel('Daily Return (%)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 5.2 Basic Statistics

In [ ]:
returns = df['Daily_Return'].dropna()

print('Daily Return Statistics:')
print(f'  Mean:   {returns.mean():.3f}%')
print(f'  Median: {returns.median():.3f}%')
print(f'  Std:    {returns.std():.3f}%')
print(f'  Min:    {returns.min():.2f}%')
print(f'  Max:    {returns.max():.2f}%')

---
## 6. More Data Exploration

### 6.1 Daily Price Range

In [ ]:
# Daily price range (High - Low)
df['Price_Range'] = df['High'] - df['Low']

plt.figure(figsize=(14, 4))
plt.plot(df.index, df['Price_Range'], color='purple', alpha=0.6, linewidth=0.5)
plt.title('Daily Price Range (High - Low)')
plt.xlabel('Date')
plt.ylabel('Price Range ($)')
plt.grid(True, alpha=0.3)
plt.show()

### 6.2 Yearly Price Comparison

In [ ]:
# Extract year and calculate yearly statistics
df['Year'] = df.index.year
yearly_stats = df.groupby('Year')['Close'].agg(['mean', 'min', 'max'])

plt.figure(figsize=(14, 5))
plt.bar(yearly_stats.index, yearly_stats['mean'], color='steelblue', alpha=0.7, label='Average Price')
plt.errorbar(yearly_stats.index, yearly_stats['mean'], 
             yerr=[yearly_stats['mean']-yearly_stats['min'], yearly_stats['max']-yearly_stats['mean']], 
             fmt='none', color='black', capsize=3, label='Min-Max Range')
plt.title('Yearly Average Stock Price with Range')
plt.xlabel('Year')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.show()

### 6.3 Open vs Close Price

In [ ]:
# Open vs Close scatter plot
plt.figure(figsize=(8, 8))
plt.scatter(df['Open'], df['Close'], alpha=0.3, s=5, color='blue')
plt.plot([df['Open'].min(), df['Open'].max()], [df['Open'].min(), df['Open'].max()], 
         'r--', linewidth=2, label='Open = Close line')
plt.title('Open Price vs Close Price')
plt.xlabel('Open Price ($)')
plt.ylabel('Close Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 6.4 Returns by Day of Week

In [ ]:
# Returns distribution by day of week
df['DayOfWeek'] = df.index.dayofweek
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

fig, ax = plt.subplots(figsize=(10, 5))
df.boxplot(column='Daily_Return', by='DayOfWeek', ax=ax)
ax.set_xticklabels(day_names)
ax.set_title('Daily Returns Distribution by Day of Week')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Daily Return (%)')
plt.suptitle('')  # Remove automatic title
plt.grid(True, alpha=0.3)
plt.show()

---
## 7. Correlation Analysis

In [ ]:
# Correlation between key metrics
corr_cols = ['Open', 'Close', 'Volume', 'Daily_Return', 'Price_Range']

corr_matrix = df[corr_cols].dropna().corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f', 
            square=True, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

---
## 8. Summary

In [ ]:
print('=== DATASET SUMMARY ===')
print(f'Period: {df.index.min().date()} to {df.index.max().date()}')
print(f'Records: {len(df)}')
print(f'Min Price: ${df["Close"].min():.2f}')
print(f'Max Price: ${df["Close"].max():.2f}')
print(f'Latest Price: ${df["Close"].iloc[-1]:.2f}')